# Notebook D — RL at Inference (TTA): the headline results

## Setup (carried over from your already-run Notebook A + Notebook B)
Same setup cells as Notebook C, but Stage B is skipped entirely — TTA adapts the
policy around the **frozen** backbone, not a Stage-B-adjusted one.

**Fix applied here that wasn't in the original spec:** the in-domain TTA cell
(CELL D3) asserts against a variable `frozen_rmse` that was never actually defined
anywhere in the original notebook — it would have crashed with a `NameError` the
first time you ran it. A small cell computing it independently (same frozen-policy
math, done once, before the TTA loop) has been added below, labeled
**CELL D-FROZEN-CHECK**.

## Setup (from your already-run Notebook A)

In [ ]:
import torch, sys, subprocess
print("torch:", torch.__version__, "| python:", sys.version.split()[0], "| cuda:", torch.cuda.is_available())
TORCH = torch.__version__.split("+")[0]
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))
pip("torch-geometric==2.6.1", "torch-scatter", "torch-sparse", "torch-cluster",
    "-f", f"https://data.pyg.org/whl/torch-{TORCH}+cpu.html")
pip("pyyaml", "scipy", "pandas", "matplotlib", "seaborn")
print("deps OK")

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/cosmic-net"
# Audited implementation lives on fork fix/rl-pruning-symmetry (upstream main
# lags the audited commits); pin both so execution matches the reviewed code.
REPO_URL = "https://github.com/Neal-Salian/cosmic-net-f.git"
REPO_BRANCH = "fix/rl-pruning-symmetry"

# Read GitHub PAT from Kaggle Secrets
# If using kaggle_secrets:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GITHUB_PAT = user_secrets.get_secret("GITHUB_PAT")

# Clone only if repo doesn't already exist
if not os.path.exists(REPO_DIR):
    auth_url = REPO_URL.replace(
        "https://",
        f"https://x-access-token:{GITHUB_PAT}@"
    )

    # SECURITY: never let the token reach the notebook output. A failed
    # subprocess.check_call raises CalledProcessError whose message embeds the
    # full command line (i.e. the token) — re-raise a sanitized error instead.
    try:
        subprocess.check_call([
            "git", "clone",
            "--branch", REPO_BRANCH,
            auth_url,
            REPO_DIR
        ])
    except subprocess.CalledProcessError:
        raise RuntimeError(
            f"git clone failed for {REPO_URL} (token redacted) — check the PAT "
            "secret, the repo URL, and Kaggle internet access."
        ) from None
    # The clone writes the token into .git/config (remote origin URL) — remove it.
    subprocess.check_call(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])

# FIX (Sep 2026 rewire): the audited RL fixes live on fix/rl-pruning-symmetry.
# A cached REPO_DIR from an older Kaggle run would otherwise silently execute
# stale pre-fix code — always fetch + check out the branch, then scrub the PAT
# from the remote URL again (fetch writes it back into .git/config).
auth_fetch = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_PAT}@")
try:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", auth_fetch, REPO_BRANCH])
except subprocess.CalledProcessError:
    raise RuntimeError(f"git fetch failed for {REPO_URL} (token redacted) — check the PAT secret and Kaggle internet access.") from None
subprocess.check_call(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"])
subprocess.check_call(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
print("branch:", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip(),
      "| HEAD:", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip())


os.chdir(REPO_DIR)
print(os.listdir("."))

In [ ]:
# Mount uploaded data (repo already cloned privately in the previous cell)
import subprocess, os, shutil
os.chdir("/kaggle/working/cosmic-net")
print(os.listdir("."))
# Upload tng100_clustered.csv + best_model_augmented.pt as a Kaggle dataset named "cosmicnet-data"
INPUT = "/kaggle/input/datasets/nealsalian/cosmicnet-data"
os.makedirs("data/raw", exist_ok=True)
shutil.copy(f"{INPUT}/tng100_clustered.csv", "data/raw/tng100_clustered.csv")
os.makedirs("kaggle", exist_ok=True)
if os.path.exists(f"{INPUT}/best_model_augmented.pt"):
    shutil.copy(f"{INPUT}/best_model_augmented.pt", "kaggle/best_model_augmented.pt")
print("data staged:", os.path.getsize("data/raw/tng100_clustered.csv")/1e6, "MB")


In [ ]:
#Load config, force CPU-safe worker settings
import sys, yaml, torch, numpy as np
sys.path.insert(0, ".")
with open("config/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["source"] = "tng"
cfg["data"]["num_workers"] = 0         
cfg["data"]["batch_size"] = 16
cfg["model"]["mc_samples"] = 30
cfg["rls"] = {
    "policy_hidden": 64, "lr": 0.001, "entropy_coef": 0.01, "value_coef": 0.5,
    "epochs": 60, "batch_size": 32,
    "target_sparsity_start": 0.9, "target_sparsity_end": 0.4,
    "sparsity_anneal_epochs": 40,
    "w_acc": 1.0, "w_sp": 0.5, "w_conn": 1.0, "w_virial": 1.0, "w_unc": 0.5,
    "virial_anneal_start_epoch": 10, "min_keep_frac": 0.1, "seed": 42,
    # TTA ("RL at inference") — tune on VAL split only, then freeze
    "tta_lr": 1e-4, "tta_steps": 10, "tta_mc_samples": 15, "tta_patience": 3,
    "tta_target_sparsity": 0.5,
}

# Sep 2026 fixes: TTA trust-region placeholders + Stage-B keys. setdefault so a
# future config.yaml switch wins. sparsity_mode is NOT defaulted here — CELL
# 10-LOAD syncs it from Notebook B's recorded config (same as Notebook C).
# tta_kl_coef / tta_lr_decay are PLACEHOLDER starting values, NOT tuned — sweep
# on VAL on Kaggle per the tuning note before the TTA cells.
cfg["rls"].setdefault("stageb_epochs", 10)
cfg["rls"].setdefault("stageb_lr", 1e-4)
cfg["rls"].setdefault("stageb_patience", 3)
cfg["rls"].setdefault("stageb_full_tol", 0.02)
cfg["rls"].setdefault("stageb_augment", "policy")
cfg["rls"].setdefault("tta_kl_coef", 0.05)
cfg["rls"].setdefault("tta_lr_decay", 0.9)
cfg["rls"].setdefault("tta_val_tol", 0.02)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| source:", cfg["data"]["source"])

In [ ]:
# PROVENANCE (recording only - no computation is affected; no secrets are read)
import hashlib, json, platform, subprocess, time

def _md5_of(path, _blk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(_blk), b""):
            h.update(chunk)
    return h.hexdigest()

def _file_info(path):
    import os
    ex = os.path.exists(path)
    return {"path": path, "exists": ex,
            "size_bytes": os.path.getsize(path) if ex else None,
            "md5": _md5_of(path) if ex else None}

def _git_info(*args):
    try:
        return subprocess.check_output(["git", "-C", REPO_DIR, *args],
                                       text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None

def record_provenance(stage, extra=None):
    """Append one stage entry to outputs/rls/provenance.json (keyed by stage,
    so A/B/B_policy/C/D entries coexist and are never silently overwritten)."""
    import os
    entry = {
        "stage": stage,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "repo": {"head": _git_info("rev-parse", "HEAD"),
                 "branch": _git_info("rev-parse", "--abbrev-ref", "HEAD"),
                 "describe": _git_info("describe", "--always")},
        "dataset": {"tng_csv": _file_info("data/raw/tng100_clustered.csv"),
                    "checkpoint": _file_info(f"{INPUT}/best_model_augmented.pt")},
        "seed": cfg.get("seed"),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": {"available": torch.cuda.is_available(),
                 "version": str(torch.version.cuda),
                 "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
        "config_used": {"data": cfg.get("data"), "graph": cfg.get("graph"),
                        "model": cfg.get("model"), "rls": cfg.get("rls")},
    }
    if extra:
        entry.update(extra)
    os.makedirs("outputs/rls", exist_ok=True)
    prov = {}
    try:
        with open("outputs/rls/provenance.json") as f:
            prov = json.load(f)
    except Exception:
        pass
    prov[stage] = entry
    with open("outputs/rls/provenance.json", "w") as f:
        json.dump(prov, f, indent=2)
    print(f"[provenance] stage '{stage}' -> outputs/rls/provenance.json "
          f"(HEAD={entry['repo']['head']}, csv md5={entry['dataset']['tng_csv']['md5']})")

record_provenance("D")

In [ ]:
#Load halos, build graphs, split
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder, build_dataloaders
loader = get_loader(cfg)
halos = loader.load()
print("total halos:", len(halos), "| split", loader.split_data.__name__ if False else "")
train_halos, val_halos, test_halos = loader.split_data()
print(f"train={len(train_halos)} val={len(val_halos)} test={len(test_halos)}")
torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])   # AFTER split_data (it resets RNG)
train_loader, val_loader, test_loader = build_dataloaders(cfg, train_halos, val_halos, test_halos)
b0 = next(iter(test_loader))
print("graph:", b0.x.shape, b0.edge_index.shape, b0.edge_attr.shape, "y:", b0.y.shape)

In [ ]:
# Load frozen backbone and verify it predicts
from model.model import load_model
import torch
import torch_geometric.data as pyg_data

ckpt = "/kaggle/input/datasets/nealsalian/cosmicnet-data/best_model_augmented.pt"

# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------
gnn = load_model(ckpt, cfg, device)
gnn.eval()

model_device = next(gnn.parameters()).device

# ---------------------------------------------------------
# Create a valid test graph
# Use the complete graph instead of taking 2 nodes
# with arbitrary edges.
# ---------------------------------------------------------
single = pyg_data.Data(
    x=b0.x,
    edge_index=b0.edge_index,
    edge_attr=b0.edge_attr
)

# One graph containing all nodes
single.batch = torch.zeros(
    single.x.shape[0],
    dtype=torch.long
)

# Move graph to the same device as the model
single = single.to(model_device)

# ---------------------------------------------------------
# Forward pass
# ---------------------------------------------------------
with torch.no_grad():
    pred, _ = gnn(single)

print(
    "smoke prediction:", pred.item(),
    "| device:", model_device,
    "| params:", sum(p.numel() for p in gnn.parameters())
)

In [ ]:
# RL helpers — imported from the committed rls/ package (Sep 2026 rewire).
# These were previously reimplemented inline in this cell; the inline copies
# drifted from rls/ (notably: no pairwise-symmetric masking, no
# topk_scheduled mode, pre-fix TTA), so this cell is imports-only now.
# Drift guard: tests/test_notebook_hygiene.py fails CI if inline copies return.
import torch
import torch.nn as nn
import torch.nn.functional as F

from rls.policy import EdgePolicyNet, build_policy
from rls.policy_gradient import (bernoulli_logp, bernoulli_entropy,
                                 sample_actions, compute_advantages,
                                 compute_pg_loss, PolicyGradientTrainer,
                                 ValueNet)
from rls.sparsify import (hard_mask, apply_min_keep_floor, repair_connectivity,
                          symmetrize_probs, repair_symmetric,
                          final_symmetric_mask, topk_scheduled_mask, eval_mask,
                          pair_asymmetry_fraction)
from rls.train_policy import (train_policy, prepare_graphs,
                              check_curriculum_divergence,
                              _graph_physics_terms as graph_physics_terms)
from rls.rewards import (compute_rewards, relative_virial_penalty,
                         virial_ratio_pruned, label_free_reward)
from rls.stageb import fine_tune_gnn, edge_dropout_masks
from rls.tta import adapt_at_test_time, mc_std, edge_kl, tta_should_enable
from rls.evaluate import build_results_table, save_paper_plots
from rls.provenance import (record_backbone, format_backbone_label,
                            require_backbone_label)


## Precompute embeddings + RL training helpers (no training loop run here)

In [ ]:
# CELL 6: Precompute frozen node embeddings + graph contexts for ALL graphs
# (Sep 2026 rewire: uses rls.train_policy.prepare_graphs — the same adapter the
# offline trainer and scripts/multiseed_rl.py use. The hand-written prepare()
# lived here; its hasattr fallbacks are unnecessary on real TNG/CAMELS graphs,
# which always carry stellar_mass/vel_disp/pos.)
from torch_geometric.nn import global_mean_pool
import torch_geometric.data as pg
train_graphs = prepare_graphs(train_loader, gnn, device)
val_graphs = prepare_graphs(val_loader, gnn, device)
test_graphs = prepare_graphs(test_loader, gnn, device)
print("train graphs:", len(train_graphs), "| ctx dim:", train_graphs[0]["ctx"].shape)


In [ ]:
# CELL 7 (retired Sep 2026): ValueNet / compute_advantages / bernoulli_entropy
# now come from rls.policy_gradient (imported in the helpers cell above).
# This cell is intentionally empty — do not re-add inline copies
# (tests/test_notebook_hygiene.py guards against drift).

In [ ]:
# CELL 8 (retired Sep 2026): graph_physics_terms / relative_virial_penalty /
# compute_rewards now come from rls.train_policy and rls.rewards (imported in
# the helpers cell above). This cell is intentionally empty — do not re-add
# inline copies (tests/test_notebook_hygiene.py guards against drift).

In [ ]:
# CELL 9 (retired Sep 2026): the GNN adapter was defined here but never used
# by any later cell; eval cells below call the GNN inline. This cell is
# intentionally empty — do not re-add inline copies
# (tests/test_notebook_hygiene.py guards against drift).

## Load the already-trained policy (instead of retraining a third time)

In [ ]:
# CELL 10-LOAD (same as Notebook C — load the already-trained policy instead of
# training a third time)
import os as _os
rls = cfg["rls"]
# (Sep 2026 rewire: policy class comes from rls.policy — same weights format.)
policy = build_policy(cfg).to(device)
POLICY_INPUT = f"{INPUT}/policy.pt"
# FAIL-CLOSED (audit HIGH-1): the repo ships a STALE PRE-FIX policy.pt at
# outputs/rls/policy.pt — never evaluate it by accident. Upload Notebook B's
# fresh policy.pt to the cosmicnet-data dataset. The override below is for
# debugging ONLY and its metrics are NOT authoritative.
ALLOW_STALE_POLICY_FALLBACK = False
ckpt_path = POLICY_INPUT if _os.path.exists(POLICY_INPUT) else None
if ckpt_path is None and ALLOW_STALE_POLICY_FALLBACK and _os.path.exists("outputs/rls/policy.pt"):
    ckpt_path = "outputs/rls/policy.pt"
    print("=" * 70)
    print("WARNING: ALLOW_STALE_POLICY_FALLBACK=True - evaluating the LOCAL")
    print("outputs/rls/policy.pt (a PRE-FIX smoke artifact committed in the repo).")
    print("Metrics from this run MUST NOT be treated as authoritative.")
    print("=" * 70)
if ckpt_path is None:
    raise RuntimeError(
        "policy.pt not found in the cosmicnet-data dataset. Run Notebook B, "
        "download its policy.pt, upload it to the dataset (Dataset -> Settings "
        "-> New Version -> upload), attach the dataset, and rerun this notebook. "
        "(Set ALLOW_STALE_POLICY_FALLBACK=True only for non-authoritative debugging.)")
policy.load_state_dict(torch.load(ckpt_path, map_location=device))
policy.eval()
print(f"loaded trained policy from {ckpt_path}")
# Integrity: if Notebook B recorded its policy artifact in provenance.json
# (uploaded with the dataset or present locally), verify we are evaluating
# exactly that file. Absent record -> notice only, never a gate.
_prov_path = next((p for p in [f"{INPUT}/provenance.json", "outputs/rls/provenance.json"]
                   if _os.path.exists(p)), None)
_bp = None
if _prov_path:
    try:
        with open(_prov_path) as _f:
            _bp = json.load(_f).get("B_policy")
    except Exception:
        _bp = None
if _bp and _bp.get("policy_artifact", {}).get("md5"):
    _loaded_md5 = _md5_of(ckpt_path)
    if _loaded_md5 != _bp["policy_artifact"]["md5"]:
        raise RuntimeError(
            f"policy.pt integrity mismatch: loaded {_loaded_md5} but Notebook B "
            f"recorded {_bp['policy_artifact']['md5']} - wrong or stale upload.")
    print(f"[policy] integrity OK: matches Notebook B provenance "
          f"(B HEAD={_bp.get('repo', {}).get('head')})")
else:
    print("[policy] no B_policy provenance record found "
          "(older B run or provenance.json not uploaded) - integrity check skipped")


# Decode-mode sync (Sep 2026): eval_mask must use the sparsity_mode the LOADED
# policy was trained under — read it from Notebook B's recorded config, not
# this notebook's inline defaults (which would silently mis-decode a
# topk-trained policy with a 0.5 threshold).
_b_rls = (_bp or {}).get("config_used", {}).get("rls", {}) if isinstance(_bp, dict) else {}
if _b_rls.get("sparsity_mode"):
    rls["sparsity_mode"] = _b_rls["sparsity_mode"]
print(f"[sparsity_mode] eval decoding with sparsity_mode={rls.get('sparsity_mode', 'penalty')!r}" +
      (" (synced from Notebook B provenance)" if _b_rls.get("sparsity_mode") else " (local default — B provenance had no record)"))


The method: at inference, for EACH input graph, a copy of the policy takes K policy-gradient steps against a **label-free** reward (MC-dropout uncertainty reduction + sparsity + connectivity + virial), then emits the final mask. Labels are used ONLY for the final evaluation RMSE — never in the reward. FROZEN mode (K=0) is the ablation.

## TTA trust-region tuning — read before running D1–D5 on Kaggle

`tta_kl_coef` / `tta_lr_decay` in the config cell are **placeholder starting
values, NOT tuned** — no GPU/val split was available where this notebook was
rewired. Before trusting any test-set TTA number:

1. Sweep `tta_kl_coef` over `[0.01, 0.05, 0.1]` on the **validation split**.
2. Freeze the winner, then touch test **once** (project convention: tune on
   val, freeze, then test).
3. Run CELL D3-gate: if val-gated TTA is worse than frozen beyond
   `tta_val_tol`, the gate fails closed — report frozen only.


In [ ]:
# CELL D1 (retired Sep 2026): mc_std / label_free_reward now come from
# rls.tta and rls.rewards (imported in the helpers cell above); adaptation
# itself is rls.tta.adapt_at_test_time. This cell is intentionally empty —
# do not re-add inline copies (tests/test_notebook_hygiene.py guards drift).

In [ ]:
# CELL D2: adaptation comes from rls.tta.adapt_at_test_time (Sep 2026 rewire:
# trust region, per-step diagnostics, symmetric masks — the inline K-step loop
# lived here). Only the small mask->prediction helper still lives here: it is
# notebook plumbing used by the frozen-check, D3, D4, and gate cells below.
import copy, time
import torch_geometric.data as pg
def mask_to_pred(g, m):
    d = pg.Data(x=g["x"], edge_index=g["edge_index"][:, m], edge_attr=g["edge_attr"][m])
    d.batch = torch.zeros(d.x.shape[0], dtype=torch.long, device=device)
    with torch.no_grad():
        pred, _ = gnn(d)
        return pred.item()


## Fix: define `frozen_rmse` before the sanity assert in the next cell uses it

In [ ]:
# CELL D-FROZEN-CHECK: independent frozen-policy RMSE, computed once, used only
# to sanity-check CELL D3's K=0 branch below (see fix note at the top of this file).
# FIX: the original CELL D3 asserted against a variable `frozen_rmse` that was
# never defined anywhere -> guaranteed NameError on first run. This cell defines it.
frozen_preds, frozen_ys = [], []
with torch.no_grad():
    for g in test_graphs:
        p = torch.sigmoid(policy(g["edge_attr"].to(device), g["emb"].to(device),
                                 g["edge_index"].to(device), g["ctx"].to(device))).squeeze(-1)
        m = eval_mask(g["edge_index"].to(device), p, rls)
        gd = {k: v.to(device) for k, v in g.items() if isinstance(v, torch.Tensor)}
        frozen_preds.append(mask_to_pred(gd, m))
        frozen_ys.append(float(g["y"].view(-1)[0]))
frozen_rmse = float(np.sqrt(((np.array(frozen_preds) - np.array(frozen_ys)) ** 2).mean()))
print("independent frozen_rmse check:", frozen_rmse)


In [ ]:
# CELL D3: In-domain TTA — frozen (K=0) vs TTA (K ablation) on the TNG test set
# Uses test_graphs (from CELL 6 above: dicts with x/edge_index/edge_attr/y/ctx/emb/pos/...)
tta_rows = []
for K in [0, 5, 10, 20]:
    preds, fulls, ys, keeps, steps, secs, hists = [], [], [], [], [], [], []
    for g in test_graphs:
        gd = {k: v.to(device) for k, v in g.items() if isinstance(v, torch.Tensor)}
        t0 = time.time()
        if K == 0:  # FROZEN mode
            with torch.no_grad():
                p = torch.sigmoid(policy(gd["edge_attr"], gd["emb"], gd["edge_index"], gd["ctx"])).squeeze(-1)
                m = eval_mask(gd["edge_index"], p, rls)
            info = {"steps_run": 0, "reward_hist": []}
        else:
            m, info = adapt_at_test_time(policy, g, gnn, rls, device, init="offline")
        secs.append(time.time() - t0)
        preds.append(mask_to_pred(gd, m)); ys.append(float(gd["y"].view(-1)[0]))
        with torch.no_grad():
            dfull = pg.Data(x=gd["x"], edge_index=gd["edge_index"], edge_attr=gd["edge_attr"])
            dfull.batch = torch.zeros(dfull.x.shape[0], dtype=torch.long, device=device)
            pf, _ = gnn(dfull)
        fulls.append(pf.item())
        keeps.append(float(m.float().mean())); steps.append(info["steps_run"]); hists.append(info["reward_hist"])
    preds, fulls, ys = np.array(preds), np.array(fulls), np.array(ys)
    row = {"mode": "frozen" if K == 0 else "tta", "K": K,
           "rmse": float(np.sqrt(((preds - ys) ** 2).mean())),
           "fidelity": float(np.corrcoef(preds, fulls)[0, 1]),
           "keep_frac": float(np.mean(keeps)), "mean_steps": float(np.mean(steps)),
           "mean_time_s": float(np.mean(secs))}
    tta_rows.append(row)
    print(row)
tta_df = pd.DataFrame(tta_rows)
tta_df.to_csv("outputs/rls/tta_indomain.csv", index=False)
assert abs(tta_df.loc[tta_df.K == 0, "rmse"].iloc[0] - frozen_rmse) < 1e-6, \
    "K=0 must reproduce the frozen policy EXACTLY (same policy, same mask, same repair)"


In [ ]:
# CELL D3-gate: fail-closed TTA gate on a VAL subset (rls.tta.tta_should_enable)
# Check BEFORE reporting the D3 test numbers (previous cell): TTA may only be reported if
# val RMSE at cfg tta_steps is within tta_val_tol of frozen val RMSE.
# Small subset (8 val graphs) to bound GPU time; tune K / tta_kl_coef on the
# FULL val split for the headline numbers, per the tuning note above.
GATE_N = 8
_gv = val_graphs[:GATE_N]
_frozen_p, _tta_p, _ys = [], [], []
for _g in _gv:
    _gd = {k: v.to(device) for k, v in _g.items() if isinstance(v, torch.Tensor)}
    with torch.no_grad():
        _p = torch.sigmoid(policy(_gd["edge_attr"], _gd["emb"], _gd["edge_index"], _gd["ctx"])).squeeze(-1)
        _m0 = eval_mask(_gd["edge_index"], _p, rls)
    _frozen_p.append(mask_to_pred(_gd, _m0)); _ys.append(float(_gd["y"].view(-1)[0]))
    _m1, _ginfo = adapt_at_test_time(policy, _g, gnn, rls, device, init="offline")
    _tta_p.append(mask_to_pred(_gd, _m1))
import numpy as _np
_frozen_rmse_val = float(_np.sqrt(_np.mean((_np.array(_frozen_p) - _np.array(_ys)) ** 2)))
_tta_rmse_val = float(_np.sqrt(_np.mean((_np.array(_tta_p) - _np.array(_ys)) ** 2)))
_gate = tta_should_enable(_frozen_rmse_val, _tta_rmse_val, tol=rls.get("tta_val_tol", 0.02))
print(f"[tta-gate] K=cfg tta_steps ({rls.get('tta_steps')}) val frozen RMSE={_frozen_rmse_val:.4f} vs TTA RMSE={_tta_rmse_val:.4f} -> "
      f"{'ENABLED' if _gate else 'DISABLED (fail closed: do not report TTA test numbers)'}")


In [ ]:
# CELL D4: OOD TTA — TNG-trained policy on CAMELS, frozen vs TTA (THE headline)
# Requires REAL CAMELS HDF5 (synthetic fallback is NOT publishable).
# Adaptation needs NO labels — that is why TTA works here and the frozen policy cannot.
cfg2 = yaml.safe_load(open("config/config.yaml"))
cfg2["data"]["source"] = "camels"
cfg2["data"]["camels"] = {"suite": "IllustrisTNG", "simulation": "LH_0",
                          "cache_dir": "/kaggle/working/camels_cache"}
cfg2["data"]["num_workers"] = 0
try:
    loader2 = get_loader(cfg2)
    halos2 = loader2.load()
    assert getattr(loader2, "used_synthetic_fallback", False) is False, \
        "synthetic CAMELS fallback is not publishable — patch camels_loader.py " \
        "(plan Task 14 Step 6) and cache the real HDF5 first"
    gb2 = GraphBuilder(cfg2)
    graphs2 = gb2.build_graphs(halos2[:100])
    from torch_geometric.data import Batch as PyGBatch
    camels_graphs = []
    with torch.no_grad():
        for g in graphs2:
            g = g.to(device)
            gb = PyGBatch.from_data_list([g])
            emb = gnn.get_embeddings(gb, embedding_point="pre_pooling")
            ctx = global_mean_pool(emb, gb.batch)
            n = g.x.shape[0]
            camels_graphs.append({
                "x": g.x, "edge_index": g.edge_index, "edge_attr": g.edge_attr, "y": g.y,
                "ctx": ctx[0], "emb": emb,
                "stellar_mass": g.stellar_mass if hasattr(g, "stellar_mass") else torch.ones(n, device=device) * 1e10,
                "vel_disp": g.vel_disp if hasattr(g, "vel_disp") else torch.ones(n, device=device) * 100,
                "half_mass_r": g.half_mass_r if hasattr(g, "half_mass_r") else torch.ones(n, device=device) * 0.01,
                "pos": g.pos if hasattr(g, "pos") else torch.zeros(n, 3, device=device),
            })
    ood_rows = []
    for mode in ["frozen", "tta"]:
        preds, ys = [], []
        for g in camels_graphs:
            gd = {k: v for k, v in g.items() if isinstance(v, torch.Tensor)}
            if mode == "frozen":
                with torch.no_grad():
                    p = torch.sigmoid(policy(gd["edge_attr"], gd["emb"], gd["edge_index"], gd["ctx"])).squeeze(-1)
                    m = eval_mask(gd["edge_index"], p, rls)
            else:
                m, info = adapt_at_test_time(policy, g, gnn, rls, device, init="offline")
            preds.append(mask_to_pred(gd, m)); ys.append(float(gd["y"].view(-1)[0]))
        preds, ys = np.array(preds), np.array(ys)
        row = {"mode": mode, "rmse": float(np.sqrt(((preds - ys) ** 2).mean())),
               "r2": float(1 - ((preds - ys) ** 2).sum() / ((ys - ys.mean()) ** 2).sum())}
        ood_rows.append(row); print("CAMELS", row)
    pd.DataFrame(ood_rows).to_csv("outputs/rls/tta_camels.csv", index=False)
    delta = ood_rows[1]["rmse"] - ood_rows[0]["rmse"]
    print(f"HEADLINE: CAMELS OOD RMSE frozen={ood_rows[0]['rmse']:.4f} vs TTA={ood_rows[1]['rmse']:.4f} "
          f"(TTA change: {delta:+.4f} dex). This is the ABSOLUTE gap — a 'fraction of "
          "degradation recovered' claim also needs the in-domain frozen RMSE as reference.")
except Exception as e:
    print("CAMELS OOD skipped/failed:", e)
    print("If this is the synthetic fallback, do NOT put these numbers in the paper.")


In [ ]:
# CELL D5: TTA plots + reward trajectories + save
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(tta_df.K.astype(str), tta_df.rmse, color=["gray"] + ["steelblue"] * 3)
axes[0].set_xlabel("TTA steps K (0 = frozen)"); axes[0].set_ylabel("test RMSE (dex)")
axes[0].set_title("In-domain: TTA vs frozen")
axes[1].bar(tta_df.K.astype(str), tta_df.mean_time_s, color=["gray"] + ["darkorange"] * 3)
axes[1].set_xlabel("TTA steps K"); axes[1].set_ylabel("adaptation time (s/graph)")
axes[1].set_title("TTA cost")
for h in hists[:5]:
    if h: axes[2].plot(h, alpha=0.7)
axes[2].set_xlabel("TTA step"); axes[2].set_ylabel("label-free reward")
axes[2].set_title("Reward trajectories (5 graphs)")
fig.tight_layout(); fig.savefig("outputs/rls/tta_results.png", dpi=200)
shutil.make_archive("/kaggle/working/rls_outputs", "zip", "outputs/rls")
print("saved outputs/rls/tta_*.csv + tta_results.png — download rls_outputs.zip")


> **TTA pitfalls (plan Part 4):** if TTA's test RMSE is WORSE than frozen while Δunc is positive, the uncertainty reward is being gamed → raise `w_virial`/`w_conn` or lower K. Tune (K, tta_lr, w_unc) on the VAL split only. K=0 must reproduce the frozen numbers exactly.